In [1]:
from pyspark.sql.session import SparkSession

spark = (SparkSession.builder.getOrCreate())

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/12 12:57:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
left = [[1],[1],[2],[3],[None],[None]]
right = [[1],[1],[1],[2],[4],[None]]


left_df = spark.sparkContext.parallelize(left).toDF(["num"])
right_df = spark.sparkContext.parallelize(right).toDF(["num"])


left_df.show()
right_df.show()

+----+
| num|
+----+
|   1|
|   1|
|   2|
|   3|
|NULL|
|NULL|
+----+

+----+
| num|
+----+
|   1|
|   1|
|   1|
|   2|
|   4|
|NULL|
+----+



remeber NULL == NULL is false
How many row for below join

inner : 7
left : 10
right: 9
outer: 12
left_semi: 3
left_anit: 3


In [ ]:
left_df.join(right_df,['num'],"inner").show() 
## no null in join b/c spark apply isnotnull on join column

+---+
|num|
+---+
|  1|
|  1|
|  1|
|  1|
|  1|
|  1|
|  2|
+---+



In [8]:
left_df.join(right_df,['num'],"inner").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [num#10L]
   +- SortMergeJoin [num#10L], [num#11L], Inner
      :- Sort [num#10L ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(num#10L, 200), ENSURE_REQUIREMENTS, [plan_id=186]
      :     +- Filter isnotnull(num#10L)
      :        +- Scan ExistingRDD[num#10L]
      +- Sort [num#11L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(num#11L, 200), ENSURE_REQUIREMENTS, [plan_id=187]
            +- Filter isnotnull(num#11L)
               +- Scan ExistingRDD[num#11L]




In [20]:
left_df.show()
right_df.show()

left_df.join(right_df,['num'],"left").show() 

+----+
| num|
+----+
|   1|
|   1|
|   2|
|   3|
|NULL|
|NULL|
+----+

+----+
| num|
+----+
|   1|
|   1|
|   1|
|   2|
|   4|
|NULL|
+----+

+----+
| num|
+----+
|   1|
|   1|
|   1|
|   1|
|   1|
|   1|
|   2|
|   3|
|NULL|
|NULL|
+----+



In [21]:
left_df.show()
right_df.show()

left_df.join(right_df,['num'],"right").show() 

+----+
| num|
+----+
|   1|
|   1|
|   2|
|   3|
|NULL|
|NULL|
+----+

+----+
| num|
+----+
|   1|
|   1|
|   1|
|   2|
|   4|
|NULL|
+----+

+----+
| num|
+----+
|   1|
|   1|
|   1|
|   1|
|   1|
|   1|
|   2|
|   4|
|NULL|
+----+



In [25]:
left_df.show()
right_df.show()


(
    left_df
.join(right_df,['num'],"full")
.select(left_df['num'].alias("num1"), right_df['num'].alias('num2'))
.show()
)

+----+
| num|
+----+
|   1|
|   1|
|   2|
|   3|
|NULL|
|NULL|
+----+

+----+
| num|
+----+
|   1|
|   1|
|   1|
|   2|
|   4|
|NULL|
+----+

+----+----+
|num1|num2|
+----+----+
|NULL|NULL|
|NULL|NULL|
|NULL|NULL|
|   1|   1|
|   1|   1|
|   1|   1|
|   1|   1|
|   1|   1|
|   1|   1|
|   2|   2|
|   3|NULL|
|NULL|   4|
+----+----+



In [ ]:
left_df.show()
right_df.show()

## ## left data  matching from right
## remember it also deduplicate
left_df.join(right_df,['num'],"left_Semi").show() 

+----+
| num|
+----+
|   1|
|   1|
|   2|
|   3|
|NULL|
|NULL|
+----+

+----+
| num|
+----+
|   1|
|   1|
|   1|
|   2|
|   4|
|NULL|
+----+

+---+
|num|
+---+
|  1|
|  1|
|  2|
+---+



In [ ]:
left_df.show()
right_df.show()

## left data not matching from right
left_df.join(right_df,['num'],"left_anti").show() 

+----+
| num|
+----+
|   1|
|   1|
|   2|
|   3|
|NULL|
|NULL|
+----+

+----+
| num|
+----+
|   1|
|   1|
|   1|
|   2|
|   4|
|NULL|
+----+

+----+
| num|
+----+
|   3|
|NULL|
|NULL|
+----+



In [ ]:
spark.stop()

: 